# Argument Prefixing

When nesting structures in `tyro`, arguments are prefixed with the parent
field name to avoid naming collisions. For example, a field `learning_rate`
inside a field called `optimizer` becomes `--optimizer.learning-rate`.

This page covers two ways to control this prefixing behavior:

- **`tyro.conf.arg(prefix_name=False)`** — applied to a field, tells tyro
  not to prefix it with the parent's accumulated prefix.
- **`tyro.conf.arg(name="")`** — applied to a field, suppresses the field's
  own name from appearing in child prefixes.

These work for both regular arguments and subcommands. For subcommands,
there is also **`tyro.conf.subcommand(prefix_name=False)`**, which is
applied to individual types within a union.

In [ ]:
import dataclasses
from typing import Annotated, Union

import tyro

# Monkeypatch tyro.cli so --help prints without exiting.
_original_cli = tyro.cli


def _patched_cli(*args, **kwargs):
    try:
        return _original_cli(*args, **kwargs)
    except SystemExit:
        pass


tyro.cli = _patched_cli

# Force ANSI colors in output.
tyro._fmtlib._FORCE_ANSI = True

## Arguments

### Default: arguments are prefixed

Consider two nested dataclasses. The inner fields are automatically prefixed
with the outer field name:

In [ ]:
@dataclasses.dataclass
class OptimizerConfig:
    learning_rate: float = 1e-3
    weight_decay: float = 0.01


@dataclasses.dataclass
class Config:
    optimizer: OptimizerConfig = dataclasses.field(default_factory=OptimizerConfig)
    epochs: int = 10


tyro.cli(Config, args=["--help"])

usage: -c [-h] [OPTIONS]

╭─ options ─────────────────────────────────────╮
│ -h, --help    show this help message and exit │
│ --epochs INT  (default: 10)                   │
╰───────────────────────────────────────────────╯
╭─ optimizer options ───────────────────────────╮
│ --optimizer.learning-rate FLOAT               │
│               (default: 0.001)                │
│ --optimizer.weight-decay FLOAT                │
│               (default: 0.01)                 │
╰───────────────────────────────────────────────╯



### `prefix_name=False`: drop the parent's prefix

`tyro.conf.arg(prefix_name=False)` on a field means "don't prefix **me**
with the prefix inherited from my parent."

To see the effect, we need at least two levels of nesting. Below,
`prefix_name=False` on `optimizer` drops the `experiment.` prefix from
`optimizer` and its children, but `optimizer` itself still appears:

In [ ]:
@dataclasses.dataclass
class OptimizerConfig:
    learning_rate: float = 1e-3


@dataclasses.dataclass
class TrainConfig:
    optimizer: Annotated[OptimizerConfig, tyro.conf.arg(prefix_name=False)] = (
        dataclasses.field(default_factory=OptimizerConfig)
    )


@dataclasses.dataclass
class Experiment:
    train: TrainConfig = dataclasses.field(default_factory=TrainConfig)


tyro.cli(Experiment, args=["--help"])

usage: -c [-h] [--optimizer.learning-rate FLOAT]

╭─ options ───────────────────────────────────╮
│ -h, --help  show this help message and exit │
╰─────────────────────────────────────────────╯
╭─ optimizer options ─────────────────────────╮
│ --optimizer.learning-rate FLOAT             │
│             (default: 0.001)                │
╰─────────────────────────────────────────────╯



Note that `--train.optimizer.learning-rate` became `--optimizer.learning-rate`.
The `train.` prefix was dropped, but `optimizer` still appears.

### `name=""`: suppress a field's name from child prefixes

`tyro.conf.arg(name="")` suppresses the field's own name from the prefix
chain. Children are prefixed with whatever came *above* this field, skipping
the field's name entirely.

Here, `name=""` on `optimizer` removes `optimizer.` from all child
argument names:

In [ ]:
@dataclasses.dataclass
class OptimizerConfig:
    learning_rate: float = 1e-3
    weight_decay: float = 0.01


@dataclasses.dataclass
class Config:
    optimizer: Annotated[OptimizerConfig, tyro.conf.arg(name="")] = dataclasses.field(
        default_factory=OptimizerConfig
    )
    epochs: int = 10


tyro.cli(Config, args=["--help"])

usage: -c [-h] [--epochs INT] [--learning-rate FLOAT] [--weight-decay FLOAT]

╭─ options ──────────────────────────────────────────────╮
│ -h, --help             show this help message and exit │
│ --epochs INT           (default: 10)                   │
│ --learning-rate FLOAT  (default: 0.001)                │
│ --weight-decay FLOAT   (default: 0.01)                 │
╰────────────────────────────────────────────────────────╯



Now `--optimizer.learning-rate` became just `--learning-rate`.

## Subcommands

The same ideas apply to subcommands. By default, subcommand names are
prefixed with the field name:

In [ ]:
@dataclasses.dataclass
class Adam:
    learning_rate: float = 1e-3
    beta1: float = 0.9


@dataclasses.dataclass
class SGD:
    learning_rate: float = 1e-2
    momentum: float = 0.9


@dataclasses.dataclass
class Config:
    optimizer: Union[Adam, SGD]


tyro.cli(Config, args=["--help"])

usage: -c [-h] {optimizer:adam,optimizer:sgd}

╭─ options ───────────────────────────────────────────╮
│ -h, --help          show this help message and exit │
╰─────────────────────────────────────────────────────╯
╭─ subcommands ───────────────────────────────────────╮
│ (required)                                          │
│   • optimizer:adam                                  │
│   • optimizer:sgd                                   │
╰─────────────────────────────────────────────────────╯



### `subcommand(prefix_name=False)`: drop the prefix for one subcommand

`tyro.conf.subcommand(prefix_name=False)` is applied to a **type within a
union**. It drops the parent prefix from that specific subcommand's name.

Here, `Adam` opts out of the `optimizer:` prefix. `SGD` is unaffected:

In [ ]:
@dataclasses.dataclass
class Adam:
    learning_rate: float = 1e-3
    beta1: float = 0.9


@dataclasses.dataclass
class SGD:
    learning_rate: float = 1e-2
    momentum: float = 0.9


@dataclasses.dataclass
class Config:
    optimizer: Union[
        Annotated[Adam, tyro.conf.subcommand(prefix_name=False)],
        SGD,
    ]


tyro.cli(Config, args=["--help"])

usage: -c [-h] {adam,optimizer:sgd}

╭─ options ──────────────────────────────────────────╮
│ -h, --help         show this help message and exit │
╰────────────────────────────────────────────────────╯
╭─ subcommands ──────────────────────────────────────╮
│ (required)                                         │
│   • adam                                           │
│   • optimizer:sgd                                  │
╰────────────────────────────────────────────────────╯



### `name=""`: suppress the field name from all subcommand names

`tyro.conf.arg(name="")` on the union field drops the field name from
**all** subcommand names at once:

In [ ]:
@dataclasses.dataclass
class Adam:
    learning_rate: float = 1e-3
    beta1: float = 0.9


@dataclasses.dataclass
class SGD:
    learning_rate: float = 1e-2
    momentum: float = 0.9


@dataclasses.dataclass
class Config:
    optimizer: Annotated[Union[Adam, SGD], tyro.conf.arg(name="")]


tyro.cli(Config, args=["--help"])

usage: -c [-h] {adam,sgd}

╭─ options ───────────────────────────────────╮
│ -h, --help  show this help message and exit │
╰─────────────────────────────────────────────╯
╭─ subcommands ───────────────────────────────╮
│ (required)                                  │
│   • adam                                    │
│   • sgd                                     │
╰─────────────────────────────────────────────╯

